# Sequence alignments

Ancestral sequences are commonly reconstructed from a multiple sequence alignment of one sequence per species, on a single tree for every column. Between closely related species, however, incomplete lineage sorting lets the gene tree vary along the sequence. {class}`~ancestree.sources.AlignmentSource` reads aligned FASTA files, each column a site and each sequence a haploid sample, so fixed-tree and local-tree inference accept an alignment in place of a VCF, and {meth}`Inference.to_fasta() <ancestree.inference.Inference.to_fasta>` writes the reconstructed sequence back as FASTA. This guide reconstructs the ancestral sequence of a simulated alignment on a fixed tree and from inferred local trees, and grades both against the ancestral reconstruction of [IQ-TREE](https://iqtree.github.io) on the same input.

## The alignment

The alignment holds one sequence for each of the species `W`, `X`, `Y` and `Z`, simulated under the species tree `(((W,X),Y),Z)` with mutations following JC69. The branch between the split of `W` and `X` and that of `Y` is short, so incomplete lineage sorting groups `Y` with `W` or `X` along a large share of the sequence. Each file is one contig, named after the file, and positions are the 1-based alignment columns. Passing a sample name as {paramref}`AlignmentSource.reference <ancestree.sources.AlignmentSource.reference>` instead places them on the ungapped sequence of that sample, skipping the columns where it has a gap.

In [1]:
import ancestree as anc

source = anc.AlignmentSource("alignment.fa.gz")
source.contig_lengths()

INFO:ancestree.AlignmentSource: Reading 1 alignment(s) of 4 samples (100,000 columns)


{'alignment': 100000}

A gap, an `N` or an IUPAC ambiguity code is read as missing data, and monomorphic columns are sites like any other.

In [2]:
next(iter(source))

Site(chrom='alignment', pos=1, alleles=('T',), tip_alleles={'W': 'T', 'X': 'T', 'Y': 'T', 'Z': 'T'}, local_tree_handle=None, info={}, unphased=frozenset(), padded=frozenset())

## Fixed-tree inference

On a given phylogeny the alignment is scored with a {class}`~ancestree.trees.FixedTree`, as under {ref}`Species-tree inference <fixed-tree-species-tree>`. The tree below is the one IQ-TREE estimated on the species-tree topology, read as written: unrooted, with a trifurcation at its root and branch lengths in expected substitutions per site, the unit the kernel takes. Under a reversible model with a stationary root prior the posterior at a node does not depend on where the tree is rooted. The posterior is reported at the most recent common ancestor of {paramref}`ingroup_samples <ancestree.inference.FixedTreeInference.ingroup_samples>`, here `W` and `X`.

In [3]:
from pathlib import Path

tree = anc.FixedTree.from_newick(Path("alignment.treefile").read_text())
print(tree.draw_text())

                                  ┃                          
               ┏━━━━━━━━━━━━━━━━━━┻━━━━┳━━━━━━━━━━━━━━┓      
     node 5 [bl=0.001349]              ┃              ┃      
       ┏━━━━━━━┻━━━━━━━┓               ┃              ┃      
       ┃               ┃        Y [bl=0.00721]        ┃      
       ┃               ┃                              ┃      
W [bl=0.007397]        ┃                              ┃      
                       ┃                              ┃      
                X [bl=0.007445]                       ┃      
                                                      ┃      
                                               Z [bl=0.04701]



In [4]:
fixed = anc.Inference.from_fixed_tree(
    source="alignment.fa.gz", tree=tree, model=anc.JC69(), ingroup_samples=["W", "X"]
)
res_fixed = list(fixed.infer())

INFO:ancestree.AlignmentSource: Reading 1 alignment(s) of 4 samples (100,000 columns)
INFO:ancestree.FixedTreeInference: Using 2 ingroup sample(s): W, X
INFO:ancestree.FixedTreeInference: Using 2 outgroup sample(s): Y, Z
INFO:ancestree.FixedTreeInference: Inferring the ancestral allele on the fixed tree, streaming the sites
FixedTreeInference: 100000 sites [00:00, 191922.37 sites/s]
INFO:ancestree.FixedTreeInference: baseline (MajorityOutgroupInference) MAP agreement: 0.875 on 6681 polymorphic sites
INFO:ancestree.FixedTreeInference: Focal node: reported at "ingroup_mrca"


## Saving the output

{meth}`Inference.to_fasta() <ancestree.inference.Inference.to_fasta>` writes the MAP sequence at the focal node, one record per contig, with a base in lower case where its posterior probability falls below `min_confidence`, as in the Ensembl ancestral-sequence files. FASTA holds no posterior over the four bases, which {meth}`Inference.to_tsv() <ancestree.inference.Inference.to_tsv>` writes as one row per site, and {class}`~ancestree.readers.Reader` reads either back (see {doc}`io`).

In [5]:
fixed.to_fasta("ancestor.fa", posteriors=res_fixed, min_confidence=0.95)
fixed.to_tsv("ancestor.tsv.gz", posteriors=res_fixed)
print("\n".join(Path("ancestor.fa").read_text().splitlines()[:3]))

INFO:ancestree.FastaWriter: Wrote the ancestral sequence of 100,000 site(s) on 1 contig(s) to ancestor.fa
INFO:ancestree.TsvWriter: Wrote 100,000 site(s) to ancestor.tsv.gz


>alignment
TGCTGTTAACCTTACTATACTCCCGCTCCGGGGCTTGGTTCATATGAACAAGTCTTTGCG
CCCATAAATGTAGCCAGTGAGCTTAGTTAGAGCAAGGGGTGCGGCAGCGCAACTCCGTCG


In [6]:
anc.Reader("ancestor.tsv.gz").head(3)

chrom      pos  AA       A       C       G       T
alignment    1  T   0.0000  0.0000  0.0000  1.0000
alignment    2  G   0.0000  0.0000  1.0000  0.0000
alignment    3  C   0.0000  1.0000  0.0000  0.0000

On the command line the same run writes both files, each `--out` path taking its format from its extension:

```bash
ancestree fixed-tree --alignment alignment.fa.gz --tree alignment.treefile \
    --ingroup W,X --out ancestor.fa ancestor.tsv.gz
```

## Local-tree inference

A single tree treats `Y` as an outgroup to the most recent common ancestor of `W` and `X`, whereas wherever the gene tree is discordant `Y` descends from it. {class}`~ancestree.local_tree_inference.LocalTreeInference` infers the gene tree along the alignment instead, the outgroups being panel haplotypes like any other (see {doc}`local_tree_inference`). The mutation and recombination rates are those of the simulation.

In [7]:
local = anc.Inference.from_local_tree(
    source="alignment.fa.gz", mu=2.5e-7, rec_rate=2.5e-8,
    ingroup_samples=["W", "X"], outgroup_samples=["Y", "Z"]
)
res_local = list(local.infer())

INFO:ancestree.AlignmentSource: Reading 1 alignment(s) of 4 samples (100,000 columns)
INFO:ancestree.LocalTreeInference: Inferring local trees from genotypes (4 samples, window=8snp)
INFO:ancestree.LocalTreeInference: Using 2 ingroup sample(s): W, X
INFO:ancestree.LocalTreeInference: Using 2 outgroup sample(s): Y, Z
LocalTreeInference: 1 segments [00:00,  1.28 segments/s]
INFO:ancestree.LocalTreeInference: 98532 site(s) are monomorphic within the ingroup and were reported at the ingroup MRCA, where the ingroup's own allele is the answer.
INFO:ancestree.LocalTreeInference: Focal node: reported at "ingroup_mrca"


## Comparison with IQ-TREE

IQ-TREE's marginal reconstruction under JC69 on the species-tree topology was computed beforehand. Its `.state` file is a tab-separated table of the posterior probabilities of the four bases at every site and internal node, here reduced to the rows of `Node2`, the most recent common ancestor of `W` and `X`. {class}`~ancestree.readers.Reader` reads it as it reads the output of {meth}`Inference.to_tsv() <ancestree.inference.Inference.to_tsv>`.

The truth is the allele at the most recent common ancestor of `W` and `X` in the gene tree of each site, read from the simulated tree sequence. Where incomplete lineage sorting makes the gene tree discordant, that node is older than the split of `W` and `X`, and it is the node whose allele the descendants of both species inherit. Every method is graded over the variable sites, a higher MAP recovery and a lower mean Brier score being better.

In [8]:
import tskit

iqtree = anc.Reader("alignment.state.gz")
ts = tskit.load("alignment.trees")
# Alignment columns are 1-based, tree-sequence positions 0-based.
truth = {pos + 1: allele for pos, allele in anc.Grade.truth_at_focal(
    ts, ingroup_samples=["W", "X"], outgroup_samples=["Y", "Z"]).items()}

In [9]:
import pandas as pd

grades = {
    "IQ-TREE": iqtree.grade(truth),
    "fixed tree": anc.Grade(res_fixed, truth),
    "local tree": anc.Grade(res_local, truth),
}
pd.DataFrame(map(tuple, grades.values()), index=grades.keys(),
             columns=["MAP recovery", "mean Brier score", "number of sites"]).round(3)

,MAP recovery,mean Brier score,number of sites
IQ-TREE,0.986,0.022,6717
fixed tree,0.986,0.022,6717
local tree,0.995,0.009,6717


Fixed-tree mode on IQ-TREE's tree reproduces IQ-TREE's posteriors to the precision of its table, so the two grade alike. Both lose accuracy where the gene tree is discordant, which local-tree mode recovers by inferring the local tree.

In [10]:
import numpy as np

post_iqtree = np.array([[a.posterior[s] for s in anc.STATES]
                        for a in iqtree.annotations()])
assert np.abs(post_iqtree - np.array([p.values for _, p in res_fixed])).max() < 1e-4
assert grades["local tree"].brier < grades["fixed tree"].brier

In [11]:
Path("ancestor.fa").unlink(missing_ok=True)
Path("ancestor.tsv.gz").unlink(missing_ok=True)